# News Articles Sentiment Scores

This notebook contains the Natural Language Processing for both the yahoo news articles and the content of the news articles found on the web pages of the companies.

NOTE: we didn't do sentiment analysis with textblob since it is outdated

In [1]:
# import findspark
import findspark
# initialize findspark with spark directory

# import pyspark
import pyspark
# create spark contextp
sc = pyspark.SparkContext()
# create spark session
spark = pyspark.sql.SparkSession(sc)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/14 19:44:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/14 19:44:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Import dependencies
import findspark, pyspark
import os
import pandas as pd
import numpy as np
import re
import string
import spacy
from functools import reduce
from pyspark.sql import DataFrame
import umap
import umap.plot
from mpl_toolkits.mplot3d import Axes3D
import random

import matplotlib.pyplot as plt
import seaborn as sns

import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, DoubleType, NumericType, StringType
from pyspark.ml.feature import Word2Vec, StandardScaler, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array
from pyspark.ml.linalg import Vectors, VectorUDT

from pyspark.ml import Pipeline

from textblob import TextBlob

from string import punctuation
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark.sql.functions import col, rand, when, avg, count, round
import psutil, os, time, pandas as pd, matplotlib.pyplot as plt

from sklearn.manifold import TSNE 
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity

# Initialize findspark to locate Spark installation
#findspark.init('/Users/douwepauwels/sources/spark')

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compiling a ufunc more than once for the same signature
  warnings.warn(msg, errors.NumbaWarning)
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/numba/np/ufunc/dufunc.py:346: NumbaWarning: Compilation requested for previously compiled argument types ((uint32,)). This has no effect and perhaps indicates a bug in the calling code (compil

In [3]:
# Create Spark1 session: we use the following configurations:

# - master("local[*]"): Run Spark locally with as many worker threads as logical cores on your machine.
# - config("spark.driver.memory", "1g"): Allocate 1 GB of memory to the Spark driver.
# - config("spark.executor.memory", "2g"): Allocate 2 GB of memory to each Spark executor.
# - appName("IntroToSpark"): Name the Spark application "IntroToSpark".

spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.ui.port", "4040") \
    .appName("IntroToSpark1") \
    .getOrCreate()

sc1 = spark.sparkContext  # To access SparkContext if needed
spark

25/12/14 19:44:59 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession

# NOTE: Assuming your SparkSession is already active and running.

FILE_PATH = "/Users/vincenzovantornout/Desktop/BIG DATA FINAL FOLDER/Big_data_group_2/Cleaned_data/biweekly_basetable.parquet"

print(f"Attempting to read Parquet file from: {FILE_PATH}")

try:
    # Load the biweekly basetable (financial data)
    biweekly_data = spark.read.parquet(FILE_PATH).cache()

    # Get the row count
    row_count = biweekly_data.count()
    
    print(f"\n Successfully loaded biweekly basetable.")
    print(f"Total rows (biweekly periods): {row_count}")
    
    # Show the first 5 rows and the schema for inspection
    print("\n--- Biweekly Basetable (Financial Features) - Showing top 5 rows ---")
    biweekly_data.show(100)
    
    print("\n--- Schema ---")
    biweekly_data.printSchema()

except Exception as e:
    print(f"\n ERROR: Failed to read the Parquet file. Ensure the path is correct and accessible.")
    raise

Attempting to read Parquet file from: /Users/vincenzovantornout/Desktop/BIG DATA FINAL FOLDER/Big_data_group_2/Cleaned_data/biweekly_basetable.parquet


25/12/14 19:45:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



 Successfully loaded biweekly basetable.
Total rows (biweekly periods): 587

--- Biweekly Basetable (Financial Features) - Showing top 5 rows ---
+------+--------------+---------------+--------------------+--------------------+---------+------------+----------+-----------+------------+-----------+----------+-------------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+------------------+----------------+------------------+------------------+--------------------+---------+--------------------+--------------------+-------+---------+---------------------+------------------------+-----------------------+--------+--------------------+-------------------------+-------------+--------------------+---------------------+---------------------+----------------------+--------------+------------------------+-------------------------+------------------------------+----------------

In [5]:
import pyspark.sql.functions as F

# NOTE: biweekly_data is the source table with financial features.

print("--- Step 1: Creating Canonical Period Lookup (The Fix) ---")

# We use MAX and MIN aggregation across ALL rows in the financial basetable 
# to define the single, widest date range for each biweekly_period, regardless of ticker/industry.
period_lookup_fixed = biweekly_data.groupBy("biweekly_period").agg(
    F.min(F.to_date(F.col("period_start"), "yyyy-MM-dd")).alias("start_date"),
    F.max(F.to_date(F.col("period_end"), "yyyy-MM-dd")).alias("end_date")
).cache()

print("\nFixed Period Lookup Table (Top 5 - Showing single date range per period):")
period_lookup_fixed.show(5, truncate=False)

--- Step 1: Creating Canonical Period Lookup (The Fix) ---

Fixed Period Lookup Table (Top 5 - Showing single date range per period):
+---------------+----------+----------+
|biweekly_period|start_date|end_date  |
+---------------+----------+----------+
|P13            |2024-07-15|2024-07-26|
|P30            |2025-03-10|2025-03-21|
|P23            |2024-12-02|2024-12-13|
|P33            |2025-04-21|2025-05-02|
|P2             |2024-02-12|2024-02-23|
+---------------+----------+----------+
only showing top 5 rows


In [6]:
import pyspark.sql.functions as F

# NOTE: Assuming period_lookup_fixed is the DataFrame from the previous step.

print("--- Verifying Uniqueness of Canonical Period Lookup ---")

# 1. Count the total number of rows in the fixed lookup table
total_rows = period_lookup_fixed.count()

# 2. Count the number of distinct biweekly_period values
distinct_periods = period_lookup_fixed.select("biweekly_period").distinct().count()

print(f"\nTotal Rows in Lookup Table: {total_rows}")
print(f"Total Unique biweekly_period values: {distinct_periods}")

if total_rows == distinct_periods:
    print("\n Verification SUCCESS: Every biweekly_period appears exactly once.")
    print("The canonical date range definition is robust.")
else:
    # If this fails, something is fundamentally wrong with the groupBy logic.
    print("\n Verification FAILED: The number of rows does not match the number of distinct periods.")
    print("Please inspect period_lookup_fixed for duplicate biweekly_period values.")

# Optional: Find any potential duplicates (should return 0 rows if successful)
potential_duplicates = period_lookup_fixed.groupBy("biweekly_period").agg(
    F.count("*").alias("count")
).filter(F.col("count") > 1)

print(f"\nRows with potential duplicates (should be 0): {potential_duplicates.count()}")

--- Verifying Uniqueness of Canonical Period Lookup ---

Total Rows in Lookup Table: 46
Total Unique biweekly_period values: 46

 Verification SUCCESS: Every biweekly_period appears exactly once.
The canonical date range definition is robust.

Rows with potential duplicates (should be 0): 0


Load in the financial times data

In [7]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType

# NOTE: Define the path to your raw news data (assuming it's loaded in your environment)
PATH_NEWS = "/Users/vincenzovantornout/Desktop/BIG DATA FINAL FOLDER/Big_data_group_2/Data/Textual_data/Financial_times_data"
# You MUST load external_ft_news before this block if it's not already loaded
external_ft_news = spark.read.json(PATH_NEWS).cache() 

print("\n--- Step 2: News Article Date Processing and Mapping ---")

# 2a. Date Conversion: Extract and convert the date from the news articles.
ft_news_processed = external_ft_news.withColumn(
    "article_date", 
    F.when(
        # Robust check for date existence and format length (YYYY-MM-DD)
        (F.col("date").isNotNull()) & (F.length(F.trim(F.col("date"))) >= 10),
        F.to_date(F.substring(F.col("date"), 1, 10), "yyyy-MM-dd")
    ).otherwise(F.lit(None).cast(DateType()))
).filter(F.col("article_date").isNotNull()).cache()

print(f"Articles after date filtering: {ft_news_processed.count()}")

# 2b. Hard Filter (Inner Join): Join news to the canonical periods.
# This assigns a 'biweekly_period' to every article based on its date falling within the canonical window.
ft_news_period_mapped = ft_news_processed.join(
    period_lookup_fixed, 
    (ft_news_processed["article_date"] >= period_lookup_fixed["start_date"]) & 
    (ft_news_processed["article_date"] <= period_lookup_fixed["end_date"]),
    "inner" 
).select(
    # Select only the columns strictly needed for the next (labeling) step
    "biweekly_period", 
    "content", 
    "title", 
    "url"
).cache()

print(f"\n Step 2 Complete: Mapped articles ready for labeling: {ft_news_period_mapped.count()} rows.")
# ft_news_period_mapped is now ready for your keyword labeling logic.


--- Step 2: News Article Date Processing and Mapping ---
Articles after date filtering: 299

 Step 2 Complete: Mapped articles ready for labeling: 237 rows.


In [8]:
import pyspark.sql.functions as F

# --- HYPER-ROBUST KEYWORD DEFINITIONS (FINAL VERSION) ---
# NOTE: The \b word boundary marker ensures that short acronyms (cu, ag, au)
# and common general terms (mine, metal, ore) are only matched as whole words,
# drastically reducing false positives.

# 1. Copper:
COPPER_KEYWORDS = r"copper|\bcu\b|cuprum"

# 2. Silver:
SILVER_KEYWORDS = r"silver|\bag\b|white metal|white-metal|silverware|silverines|argentum"

# 3. Gold:
GOLD_KEYWORDS = r"gold|\bau\b|yellow metal|bullion|gold rush|gold-plated|sovereign|fineness|aurum"

# 4. Other Precious Metals & Mining:
OTHER_PRECIOUS_KEYWORDS = r"platinum|palladium|rhodium|iridium|osmium|ruthenium|\bpgm\b|precious metal(s)"

# 5. Other Industrial Metals & Mining:
OTHER_INDUSTRIAL_KEYWORDS = r"lead|zinc|tin|nickel|aluminum|bismuth|indium|germanium|tungsten|antimony|lithium|cobalt|rare earth(s)|critical mineral(s)|\bmine\b|\bmining\b|\bminer\b|\bminerals\b|metallurgical|\bmetal(s)\b|\bore(s)\b"

# --- Define the CRUCIAL Helper function for robust Title OR Content check ---
def check_keyword(keyword):
    """Checks if the keyword pattern is present in either the lowercased content OR title."""
    return (F.lower(F.col("content")).rlike(keyword)) | (F.lower(F.col("title")).rlike(keyword))



In [9]:
import pyspark.sql.functions as F

# Assuming keywords and check_keyword are defined...

# --- RERUN STEP 2: APPLYING MULTI-ASSIGNMENT HIERARCHY ---

# 4. Apply Multi-Assignment Labeling (Specific Metals can co-exist; General are Exclusive)

# Start with a temporary DataFrame holding the raw keyword matches (same as before)
ft_news_temp = ft_news_period_mapped.withColumn(
    "match_gold", check_keyword(GOLD_KEYWORDS).cast("int")
).withColumn(
    "match_silver", check_keyword(SILVER_KEYWORDS).cast("int")
).withColumn(
    "match_copper", check_keyword(COPPER_KEYWORDS).cast("int")
).withColumn(
    "match_other_precious", check_keyword(OTHER_PRECIOUS_KEYWORDS).cast("int")
).withColumn(
    "match_other_industrial", check_keyword(OTHER_INDUSTRIAL_KEYWORDS).cast("int")
)

# Now apply the hierarchy:
ft_news_labeled = ft_news_temp.withColumn(
    # 1. Gold, Silver, Copper: Assigned independently if keywords match.
    "focus_gold", F.col("match_gold")
).withColumn(
    "focus_silver", F.col("match_silver")
).withColumn(
    "focus_copper", F.col("match_copper")
).withColumn(
    # 4. Other Precious: Assigned only if match AND none of the top 3 specific metals were mentioned.
    "focus_other_precious", F.when(
        (F.col("match_other_precious") == 1) & 
        (F.col("focus_gold") == 0) & 
        (F.col("focus_silver") == 0) & 
        (F.col("focus_copper") == 0),
        1
    ).otherwise(0)
).withColumn(
    # 5. Other Industrial: Assigned only if match AND none of the categories (1-4) were set.
    "focus_other_industrial", F.when(
        (F.col("match_other_industrial") == 1) & 
        (F.col("focus_gold") == 0) & 
        (F.col("focus_silver") == 0) & 
        (F.col("focus_copper") == 0) & 
        (F.col("focus_other_precious") == 0),
        1
    ).otherwise(0)
).cache()

# Drop temporary 'match_X' columns
ft_news_labeled = ft_news_labeled.drop(*[col for col in ft_news_labeled.columns if col.startswith('match_')])

print("\n Step 4: Multi-Assignment Hierarchy (Specific metals can overlap; General are exclusive) Applied.")

# You must now run the next two steps (Block 1 and Block 2 from the previous response) 
# to finalize the tables with this new logic.


 Step 4: Multi-Assignment Hierarchy (Specific metals can overlap; General are exclusive) Applied.


In [10]:
import pyspark.sql.functions as F

print("\n--- STEP 4: Wide-to-Long Transformation (Converts 1 row/article to N rows/assignment) ---")

# Define the final industry map
INDUSTRY_COLS = [
    ("Copper", "focus_copper"),
    ("Silver", "focus_silver"),
    ("Gold", "focus_gold"),
    ("Other Precious Metals & Mining", "focus_other_precious"),
    ("Other Industrial Metals & Mining", "focus_other_industrial")
]

# 1. Transform the 'ft_news_labeled' table to Long Format using F.explode
# This structure is common in Big Data pipelines 
ft_news_long = ft_news_labeled.select(
    "biweekly_period", "content", "title", "url", 
    F.explode(
        F.array([
            F.struct(F.lit(name).alias("industry"), F.col(col_name).alias("is_focus"))
            for name, col_name in INDUSTRY_COLS
        ])
    ).alias("industry_focus")
).cache()

# 2. Filter for positive focus (where focus_X was 1) and select final columns.
# This creates the 'industry' column that your verification code was looking for.
final_ft_news_base = ft_news_long.where(F.col("industry_focus.is_focus") == 1).select(
    "biweekly_period", 
    F.col("industry_focus.industry").alias("industry"), # <-- **The column is created here**
    "content", 
    "title",   
    "url"
).cache()

print(" Step 4 Complete: News data is now in the final long format (final_ft_news_base).")
print(f"Total ROWS (now containing the 'industry' column): {final_ft_news_base.count()}")


--- STEP 4: Wide-to-Long Transformation (Converts 1 row/article to N rows/assignment) ---
 Step 4 Complete: News data is now in the final long format (final_ft_news_base).


Total ROWS (now containing the 'industry' column): 279


In [11]:
import pyspark.sql.functions as F

# We check the article that caused issues (Silver price hits record...)
KNOWN_URL = "https://www.ft.com/content/e1eb2205-bba5-4b66-888b-e7e5d6dc45f4"

print("--- Step 1: Overall Assignment Verification ---")

final_unique_article_count = final_ft_news_base.select("url").distinct().count()
total_rows_in_long_format = final_ft_news_base.count()

print(f"Total UNIQUE articles remaining: {final_unique_article_count}")
print(f"Total ROWS (reflects multi-industry articles): {total_rows_in_long_format}")

if final_unique_article_count > 0:
    avg_assignments = total_rows_in_long_format / final_unique_article_count
    print(f"Average assignments per article: {avg_assignments:.2f}")


print("\n--- Step 2: Verification of the Problematic Article (Gold/Silver) ---")
specific_article_assignments = final_ft_news_base.filter(F.col("url") == KNOWN_URL).select(
    "biweekly_period", 
    "industry", 
    "title",
    "content"
    
).distinct().show(truncate=False)

--- Step 1: Overall Assignment Verification ---
Total UNIQUE articles remaining: 219
Total ROWS (reflects multi-industry articles): 279
Average assignments per article: 1.27

--- Step 2: Verification of the Problematic Article (Gold/Silver) ---
+---------------+--------+-------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
import pyspark.sql.functions as F

# NOTE: This assumes final_ft_news_base is the result of Block 1 (Wide-to-Long Rerun)

print("--- Verification: Top 5 Multi-Focus Articles ---")

# 1. Group by URL and count the number of industries assigned to each article
multi_assignment_check = final_ft_news_base.groupBy("url", "title").agg(
    F.count("industry").alias("industry_count")
).orderBy(F.desc("industry_count")).cache()

print(f"\nTotal articles assigned to multiple industries: {multi_assignment_check.filter(F.col('industry_count') > 1).count()}")

print("\nTop 5 articles assigned to the most industries (should ideally be 2 or 3):")
multi_assignment_check.show(5, truncate=False)

# 2. Get the URL of the article with the maximum assignments (to check its specific assignments)
top_url = multi_assignment_check.head()["url"]

print(f"\nSpecific assignments for the top multi-focus article (URL: {top_url}):")
final_ft_news_base.filter(F.col("url") == top_url).select(
    "biweekly_period", 
    "industry", 
    "title", 
    "content"
).distinct().show(truncate=False)

# The output for the 'industry' column in Step 2 should ideally only contain 
# 'Copper', 'Silver', and/or 'Gold'. If it contains 'Other Precious Metals...' 
# or 'Other Industrial Metals...', the hierarchy is still failing or your 
# specific metal keywords need further refinement.

--- Verification: Top 5 Multi-Focus Articles ---

Total articles assigned to multiple industries: 53

Top 5 articles assigned to the most industries (should ideally be 2 or 3):
+---------------------------------------------------------------+------------------------------------------------------------------------+--------------+
|url                                                            |title                                                                   |industry_count|
+---------------------------------------------------------------+------------------------------------------------------------------------+--------------+
|https://www.ft.com/content/1ca28611-5ed7-4b58-bfe3-37dc84d321a9|Royal Mint strikes gold with e-waste recovery facility                  |3             |
|https://www.ft.com/content/9d48a9d1-d699-4dbb-9978-77b81157db20|US metals prices soar to big premiums ahead of Donald Trump’s tariffs   |3             |
|https://www.ft.com/content/1a267b09-0238-4da1-b5db-4

In [13]:

import pyspark.sql.functions as F

# NOTE: This code assumes 'biweekly_data' (financial source) and 'final_ft_news_base' (clean news) 
# are loaded and active in your Spark session.

print("\n--- Final Raw Text Merge: Linking All Relevant Articles to Sector/Period ---")

# 1. Prepare the Financial Data Keys (Sector/Period View)
# We extract only the unique period and industry keys from the financial table.
# This explicitly drops the 'ticker' and other company-specific features, as requested.
sector_period_data = biweekly_data.select(
    "biweekly_period",
    "industry",
    F.to_date(F.col("period_start"), "yyyy-MM-dd").alias("period_start"),
    F.to_date(F.col("period_end"), "yyyy-MM-dd").alias("period_end")
).distinct().cache()

# Optional: Print count of unique financial keys
print(f"Total unique financial keys (Period + Industry): {sector_period_data.count()}")


# 2. Perform the Inner Join
# This links the unique sector/period combinations from the financial data 
# to every relevant article from the news data.
final_merged_data_raw = sector_period_data.join(
    final_ft_news_base,
    on=["biweekly_period", "industry"], 
    how="inner" 
).cache()

print(" Final Merge Complete. Merged table is structured for Sentiment Analysis UDFs.")
print(f"Total rows in final merged table: {final_merged_data_raw.count()}")

final_merged_data_raw.show(5, truncate=False)
final_merged_data_raw.printSchema()



--- Final Raw Text Merge: Linking All Relevant Articles to Sector/Period ---
Total unique financial keys (Period + Industry): 237
 Final Merge Complete. Merged table is structured for Sentiment Analysis UDFs.
Total rows in final merged table: 319
+---------------+--------+------------+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
import pyspark.sql.functions as F

print("\n--- Final Raw Text Merge: CORRECTED JOIN KEY ---")

# 1. Prepare the Financial Data Keys (Sector/Period View)
# IMPORTANT: We only select the actual join keys to ensure perfect uniqueness.
# We will pull the canonical dates from the FIXED LOOKUP LATER.
sector_period_data_keys = biweekly_data.select(
    "biweekly_period",
    "industry"
).distinct().cache()

print(f"Total unique JOIN keys (Period + Industry): {sector_period_data_keys.count()}")

# 2. Perform the Inner Join
final_merged_data_raw = sector_period_data_keys.join(
    final_ft_news_base,
    on=["biweekly_period", "industry"], 
    how="inner" 
).cache()

print("\n FINAL MERGE COMPLETE: Synchronization Achieved.")
print(f"Total rows in final merged table: {final_merged_data_raw.count()}")


--- Final Raw Text Merge: CORRECTED JOIN KEY ---
Total unique JOIN keys (Period + Industry): 206

 FINAL MERGE COMPLETE: Synchronization Achieved.
Total rows in final merged table: 271


In [15]:
import pyspark.sql.functions as F

print("--- 1. Schema Check for final_merged_data_raw ---")
final_merged_data_raw.printSchema()

# We will then use the column names from this output to build the next select statement.

--- 1. Schema Check for final_merged_data_raw ---
root
 |-- biweekly_period: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- content: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)



In [16]:
import pyspark.sql.functions as F

# NOTE: Assuming final_merged_data_raw is the result of the final join (271 rows)

print("--- 1. Schema and Total Row Count ---")
print(f"Total Rows in Final Merged Table: {final_merged_data_raw.count()}")
final_merged_data_raw.printSchema()

print("\n--- 2. Data Sample (Focus on Keys and Content) ---")
# Select key columns and truncate the long content for easy viewing
final_merged_data_raw.select(
    "biweekly_period",
    "industry",
    "title",
    "content",
    "url"
).show(10, truncate=False)

print("\n--- 3. Industry Distribution Check ---")
# Verify how many articles were assigned to each industry
final_merged_data_raw.groupBy("industry").agg(
    F.count("*").alias("article_count"),
    F.countDistinct("url").alias("unique_articles")
).orderBy(F.desc("article_count")).show(truncate=False)

--- 1. Schema and Total Row Count ---
Total Rows in Final Merged Table: 271
root
 |-- biweekly_period: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- content: string (nullable = true)
 |-- title: string (nullable = true)
 |-- url: string (nullable = true)


--- 2. Data Sample (Focus on Keys and Content) ---
+---------------+--------+---------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Punctuation, stopwords and other preprocessing

These are all the functions needed for preprocessing the data + A List of all the content preprocessing functions

In [17]:
# -----------------------
# REMOVE TEXT FUNCTIONS
# -----------------------

# define punctuation and stopwords
PUNCTUATION = [char for char in punctuation if char not in ["!", "@", "#", "%", "(", ")"]] # ADD more if needed
STOPWORDS = stopwords.words("english")
CONTEXT_WORDS = ['gold', 'silver', 'copper']

# define function to remove punctuation
def remove_punct(text):
    # remove punctuation
    text = "".join([char for char in text if char not in PUNCTUATION])
    return(text)

# define function to remove stopwords
def remove_stopwords(text):
    text = " ".join([word for word in text.split(" ") if word not in STOPWORDS])
    return(text)

# define function to remove context words
def remove_context_words(text):
    text = " ".join([word for word in text.split(" ") if word not in CONTEXT_WORDS])
    return(text)


# define function to remove urls
def remove_urls(text):
    url_pattern = r'http\S+|www\.\S+'
    return re.sub(url_pattern, '', text)

# define function to remove numbers 
def remove_numbers(text):
    pattern = r'\b\d+(\.\d+)?([kKmMbB])?\b|\$\d+(\.\d+)?'
    return re.sub(pattern, '', text)

# define function to remove stock tickers 
def remove_tickers(text):
    # tickers like AEM.TO, AAA.V, XYZ.CD
    ticker_pattern = r'\b[A-Z]{2,5}\.[A-Z]{1,3}\b'
    # tickers like AAPL, TSLA
    plain_ticker_pattern = r'\b[A-Z]{2,5}\b'
    # cashtags like $AAPL
    cashtag_pattern = r'\$[A-Za-z]{2,5}'

    text = re.sub(ticker_pattern, '', text)
    text = re.sub(cashtag_pattern, '', text)
    text = re.sub(plain_ticker_pattern, '', text)
    return text

# define function to remove unnecassary whitespaces
def clean_whitespace(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

# define function to lemmatize text
nlp = spacy.load("en_core_web_lg")

def spacy_lemmatize(text):
    doc = nlp(text)
    return " ".join([token.lemma_ for token in doc])

#...
# 

# -----------------------
# Removing part of the text: general company info after each article, licence info, etc.
# -----------------------
# For now wait with this since we have data from different sources within yahoo and the news websites 



# -----------------------
# DON'T forget to register them as a udf 
# -----------------------
remove_punct_udf = F.udf(remove_punct, StringType())
remove_stopwords_udf = F.udf(remove_stopwords, StringType())
remove_context_words_udf = F.udf(remove_context_words, StringType())
remove_urls_udf = F.udf(remove_urls, StringType())
remove_numbers_udf = F.udf(remove_numbers, StringType())
remove_tickers_udf = F.udf(remove_tickers, StringType())
clean_whitespace_udf = F.udf(clean_whitespace, StringType())
spacy_lemmatize_udf = F.udf(spacy_lemmatize, StringType())

In [18]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# NOTE: This block relies on all your UDFs (remove_urls_udf, remove_punct_udf, etc.) 
# already being defined and registered in the active session.

def clean_df_for_sentiment(df):
    """
    Applies the full text cleaning pipeline to the content and title columns,
    creating new 'cleaned_' and 'lemma' columns for sentiment analysis.
    
    Args:
        df (pyspark.sql.DataFrame): The input DataFrame (e.g., final_merged_data_raw).
    
    Returns:
        pyspark.sql.DataFrame: DataFrame with all cleaned and feature columns added.
    """
    
    # 1. Initialize new columns by lowercasing the originals
    cleaned_df = df \
        .withColumn("cleaned_content", F.lower(F.col("content"))) \
        .withColumn("cleaned_title", F.lower(F.col("title")))

    # 2. Sequential Cleaning Steps (applied to the new 'cleaned_' columns)
    cleaned_df = cleaned_df \
        .withColumn("cleaned_content", remove_urls_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_urls_udf("cleaned_title")) \
        .withColumn("cleaned_content", remove_numbers_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_numbers_udf("cleaned_title")) \
        .withColumn("cleaned_content", remove_tickers_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_tickers_udf("cleaned_title")) \
        .withColumn("cleaned_content", remove_punct_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_punct_udf("cleaned_title")) \
        .withColumn("cleaned_content", remove_stopwords_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_stopwords_udf("cleaned_title")) \
        .withColumn("cleaned_content", remove_context_words_udf("cleaned_content")) \
        .withColumn("cleaned_title", remove_context_words_udf("cleaned_title")) \
        .withColumn("cleaned_content", F.regexp_replace("cleaned_content", "\n", " ")) \
        .withColumn("cleaned_title", F.regexp_replace("cleaned_title", "\n", " ")) \
        .withColumn("cleaned_content", clean_whitespace_udf("cleaned_content")) \
        .withColumn("cleaned_title", clean_whitespace_udf("cleaned_title"))

    # 3. Lemmatization (Create lemma columns for robust analysis)
    cleaned_df = cleaned_df \
        .withColumn("content_lemma", spacy_lemmatize_udf("cleaned_content")) \
        .withColumn("title_lemma", spacy_lemmatize_udf("cleaned_title"))

    # 4. Tokenization and Word Count (Create token lists and feature columns)
    cleaned_df = cleaned_df \
        .withColumn("title_tokenized", F.split("cleaned_title", " ")) \
        .withColumn("content_tokenized", F.split("cleaned_content", " ")) \
        .withColumn("num_words_title", F.size("title_tokenized")) \
        .withColumn("num_words_content", F.size("content_tokenized"))

    return cleaned_df


# --- EXECUTE THE CLEANING FUNCTION ---
print("--- Applying Centralized Text Cleaning Function ---")

# Run the cleaning function on your final raw data
final_merged_data_clean = clean_df_for_sentiment(final_merged_data_raw).cache()

print(" Text Cleaning Pipeline Complete. New DataFrame: final_merged_data_clean")

# Verification: Show the final feature columns
final_merged_data_clean.select(
    "biweekly_period",
    "industry",
    "num_words_content",
    F.substring(F.col("content"), 1, 50).alias("raw_content_preview"),
    F.substring(F.col("content_lemma"), 1, 50).alias("lemma_preview") # Show the final lemma output
).show(5, truncate=False)

--- Applying Centralized Text Cleaning Function ---
 Text Cleaning Pipeline Complete. New DataFrame: final_merged_data_clean


+---------------+--------+-----------------+--------------------------------------------------+--------------------------------------------------+
|biweekly_period|industry|num_words_content|raw_content_preview                               |lemma_preview                                     |
+---------------+--------+-----------------+--------------------------------------------------+--------------------------------------------------+
|P45            |Silver  |292              |Metal has staged a bigger rally than gold this yea|metal stag big rally year silver price hit record |
|P45            |Gold    |292              |Metal has staged a bigger rally than gold this yea|metal stag big rally year silver price hit record |
|P38            |Silver  |251              |Its use in growing sectors such as green tech, ele|use grow sector green tech electronic defence mean|
|P38            |Gold    |251              |Its use in growing sectors such as green tech, ele|use grow sector green t

## Sentiment analysis using TextBlob

functions to retrieve the sentiment scores

In [19]:
# -----------------------
# POLARITY AND SUBJECTIVITY
# -----------------------
def get_polarity(row):
    textBlob_review = TextBlob(row)
    return textBlob_review.sentiment[0]

def get_subjectivity(row):
    textBlob_review = TextBlob(row)
    return textBlob_review.sentiment[1]

# register functions as udf
get_polarity_udf = F.udf(get_polarity, DoubleType())
get_subjectivity_udf = F.udf(get_subjectivity, DoubleType())

def main(list_dfs):

    dfs_with_sentiment = []

    for df in list_dfs:
        df_with_sentiment = df.withColumn("polarity_title", get_polarity_udf(F.col("title_lemma"))) \
                              .withColumn("polarity_content", get_polarity_udf(F.col("content_lemma"))) \
                              .withColumn("subjectivity_title", get_subjectivity_udf(F.col("title_lemma"))) \
                              .withColumn("subjectivity_content", get_subjectivity_udf(F.col("content_lemma"))) \
                              .filter("num_words_title > 0 OR num_words_content > 0") \
                              .drop("url") \
                              .cache()
                            

        dfs_with_sentiment.append(df_with_sentiment)
        
    return dfs_with_sentiment

In [21]:
if __name__ == "__main__":
    # Make sure detijd_final is a list (wrap it if it's a single DataFrame)
    if not isinstance(final_merged_data_clean, list):
        final_merged_data_clean = [final_merged_data_clean]
    
    dfs = main(final_merged_data_clean)
    
    dfs[0].show()



25/12/14 19:47:56 WARN CacheManager: Asked to cache already cached data.


+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+
|biweekly_period|            industry|             content|               title|     cleaned_content|       cleaned_title|       content_lemma|         title_lemma|     title_tokenized|   content_tokenized|num_words_title|num_words_content|polarity_title|    polarity_content| subjectivity_title|subjectivity_content|
+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+
|            P45|              Silver|Metal ha

## Vector Embeddings using FinBERT

In [22]:
#To install torch with cuda support, follow instructions at https://pytorch.org/get-started/locally/. This will ensure that the model runs on GPU if available.
import torch

from pyspark.ml.functions import array_to_vector
from transformers import AutoTokenizer, AutoModel

Function list for embedding the text

In [23]:
# -----------------------
# INIT MODEL AND TOKENIZER
# -----------------------
MODEL_NAME = "ProsusAI/finbert"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

_tokenizer = None
_model = None

def get_model_and_tokenizer():
    global _tokenizer, _model
    if _tokenizer is None or _model is None:
        _tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        _model = AutoModel.from_pretrained(MODEL_NAME)
        _model.to(device)
        _model.eval()
    return _tokenizer, _model


# -----------------------
# EMBEDDING TEXT
# -----------------------
def get_embeddings(text):
    if text is None or text == "":
        return None
    
    tokenizer, model = get_model_and_tokenizer()
    
    # Tokenize the input text
    inputs = tokenizer(text, 
                     padding=True, 
                     truncation=True, 
                     max_length=512,
                     return_tensors="pt")
    
    # Move inputs to device
    inputs = {key: val.to(device) for key, val in inputs.items()}
    
    # Generate embeddings without gradient computation
    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling - take attention mask into account for correct averaging so we don't include padding tokens
    attention_mask = inputs['attention_mask']
    token_embeddings = outputs.last_hidden_state
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # Convert to numpy and then to list
    embedding_list = embeddings.cpu().numpy()[0].tolist()
    
    return embedding_list

# Register the function as a UDF
get_embeddings_udf = F.udf(get_embeddings, ArrayType(DoubleType()))

# -----------------------
# MAIN 
# -----------------------
def main(list_dfs):

    dfs_with_embeddings = [] 

    for df in list_dfs:

        # APPLY EMBEDDING
        df_with_embeddings = df.withColumn(
            "finbert_embeddings", 
            get_embeddings_udf(F.col("content"))
        )

        # VECTOR ARRAY
        df_with_embeddings = df_with_embeddings.filter(F.col("finbert_embeddings").isNotNull()) \
                                               .withColumn("finbert_embeddings", array_to_vector(F.col("finbert_embeddings"))) \
                                               .cache()
    
        # APPEND 
        dfs_with_embeddings.append(df_with_embeddings)

    # RETURN 
    return dfs_with_embeddings

In [26]:
if __name__ == "__main__":
    # Make sure detijd_final is a list (wrap it if it's a single DataFrame)
    if not isinstance(dfs, list):
        dfs = [dfs]
    
    dfs1 = main(dfs)
    
    dfs1[0].show()

25/12/14 19:48:49 WARN CacheManager: Asked to cache already cached data.


+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+--------------------+
|biweekly_period|            industry|             content|               title|     cleaned_content|       cleaned_title|       content_lemma|         title_lemma|     title_tokenized|   content_tokenized|num_words_title|num_words_content|polarity_title|    polarity_content| subjectivity_title|subjectivity_content|  finbert_embeddings|
+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+-----

## Sentiment Analysis using Ollama

In [27]:
# ----------------------------------------------------
# INIT MODEL NAME
# ----------------------------------------------------
import ollama
MODEL_NAME = 'gemma3:4b'


# ----------------------------------------------------
# OLLAMA 
# ----------------------------------------------------
# Function to get sentiment using ollama
def get_ollama_sentiment(text):
    try:

        prompt = f"""Provide the sentiment of the following text as a single numerical value between -1 and 1, 
        where -1 is very negative, 0 is neutral, and 1 is very positive, from an investment perspective. Only provide the numerical value, 
        I don't want any explanation or other text answer.\n\nText: '{text}'"""
        
        response = ollama.chat(model=MODEL_NAME, 
                               messages=[
                                {
                                    'role': 'user',
                                    'content': prompt
                                }
                            ])
        
        return (response['message']['content'].strip())
    except Exception as e:
        return None
    
# Register udf
get_ollama_sentiment_udf = F.udf(get_ollama_sentiment, StringType())


# ----------------------------------------------------
# MAIN: add Ollama sentiment to each article
# ----------------------------------------------------
def main(list_dfs):
    """
    Takes a list of Spark DataFrames
    
    'ollama_sentiment': string with the numeric sentiment between -1 and 1.
    """
    dfs_with_sentiment = []

    for df in list_dfs:
        df_with_sentiment = df.withColumn(
            "ollama_sentiment",
            get_ollama_sentiment_udf(F.col("content"))
        ).cache()
        dfs_with_sentiment.append(df_with_sentiment)

    return dfs_with_sentiment

In [28]:
if __name__ == "__main__":
    if not isinstance(dfs1, list):
        dfs1 = [dfs1]
    
    dfs2 = main(dfs1)
    dfs2[0].show()

+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+-------------------+--------------------+--------------------+----------------+
|biweekly_period|            industry|             content|               title|     cleaned_content|       cleaned_title|       content_lemma|         title_lemma|     title_tokenized|   content_tokenized|num_words_title|num_words_content|polarity_title|    polarity_content| subjectivity_title|subjectivity_content|  finbert_embeddings|ollama_sentiment|
+---------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+-----------------+--------------+--------------------+------------

In [32]:
#Save to a parquet file 
# Extract the DataFrame from the list
df = dfs2[0]

# Save to parquet
df.write.parquet('Financial_times_sentiment.parquet')

# Basetable 

In [ ]:
# -------------------------------------------------------
# Function that takes the average over a group of vectors
# -------------------------------------------------------

def mean_array(arrs):
    """
    arrs: list of embeddings, each embedding is a list/array of floats.
    Returns element-wise mean embedding as a Python list of floats.
    """
    if arrs is None or len(arrs) == 0:
        return None

    n = len(arrs)
    dim = len(arrs[0])

    sums = [0.0] * dim
    count = 0

    for a in arrs:
        if a is None:
            continue
        count += 1
        for i, v in enumerate(a):
            sums[i] += float(v)

    if count == 0:
        return None

    return [s / count for s in sums]

mean_array_udf = F.udf(mean_array, ArrayType(DoubleType()))

# Optional: if you want the result back as a Spark Vector
to_vec_udf = F.udf(
    lambda arr: Vectors.dense(arr) if arr is not None else None,
    VectorUDT()
)

# -------------------------------------------------------
# Main aggregation function
# -------------------------------------------------------

def aggregate_sentiment(df):
    df = df.withColumn("finbert_arr", vector_to_array("finbert_embeddings"))

    agg_sent = (
        df
        .groupBy(
            F.col("biweekly_period")
        )
        .agg(
            # DEPENDENT VARIABLE
            F.mode('next_announcement').alias("next_announcement"),

            # GENERAL ARTICLE FEATURES 
            F.count("*").alias("n_artices"),
            F.avg('num_words_title').alias("avg_num_words_title"), 
            F.avg('num_words_content').alias("avg_num_words_content"), 

            # POLARITY AND SUBJECTIVITY FEATURES
            F.avg("polarity_title").alias("avg_polarity_title"),
            F.avg("polarity_content").alias("avg_polarity_content"),
            F.avg("subjectivity_title").alias("avg_subjectivity_title"),
            F.avg("subjectivity_content").alias("avg_subjectivity_content"),

            # OLLAMA SENTIMENT FEATURES 
            F.avg("ollama_sentiment").alias("avg_ollama"),
            F.stddev_pop("ollama_sentiment").alias("std_ollama"),
            F.min("ollama_sentiment").alias("min_ollama"), 
            F.max("ollama_sentiment").alias("max_ollama"), 

            # EMBEDDING FEATURES ...
            F.collect_list("finbert_arr").alias("finbert_arr_list")
        )
    )

    agg_sent = (
        agg_sent
        .withColumn("avg_finbert_array", mean_array_udf("finbert_arr_list"))
        .drop("finbert_arr_list")
    )

    agg_sent = agg_sent.withColumn(
        "avg_finbert_vec",
        to_vec_udf("avg_finbert_array")
    )
    
    return agg_sent


def main(dfs_list): 

    dfs_basetable = []

    for df in dfs_list:
        basetable = aggregate_sentiment(df).cache()

        dfs_basetable.append(basetable)

    return dfs_basetable
    

In [ ]:
if __name__ == "__main__":
    # Make sure detijd_final is a list (wrap it if it's a single DataFrame)
    if not isinstance(dfs2, list):
        dfs2 = [dfs2]
    
    dfs3 = main(dfs2)
    print(f"✅ Success! Processed {len(result)} DataFrames")
    dfs3[0].show()

{"ts": "2025-12-14 11:58:11.816", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `next_announcement` cannot be resolved. Did you mean one of the following? [`cleaned_content`, `content`, `ollama_sentiment`, `cleaned_title`, `content_lemma`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o2272.agg.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `next_announcement` cannot be resolved. Did you mean one of the following? [`cleaned_content`, `content`, `ollama_sentiment`, `cleaned_title`, `content_lemma`]. SQLSTATE: 42703;\n'Aggregate [biweekly_period#18853], [biweekly_

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `next_announcement` cannot be resolved. Did you mean one of the following? [`cleaned_content`, `content`, `ollama_sentiment`, `cleaned_title`, `content_lemma`]. SQLSTATE: 42703;
'Aggregate [biweekly_period#18853], [biweekly_period#18853, 'mode('next_announcement, false) AS next_announcement#33289, count(1) AS n_artices#33290L, avg(num_words_title#29874) AS avg_num_words_title#33291, avg(num_words_content#29875) AS avg_num_words_content#33292, avg(polarity_title#30326) AS avg_polarity_title#33293, avg(polarity_content#30328) AS avg_polarity_content#33294, avg(subjectivity_title#30330) AS avg_subjectivity_title#33295, avg(subjectivity_content#30332) AS avg_subjectivity_content#33296, avg(cast(ollama_sentiment#32440 as double)) AS avg_ollama#33297, stddev_pop(cast(ollama_sentiment#32440 as double)) AS std_ollama#33298, min(ollama_sentiment#32440) AS min_ollama#33299, max(ollama_senti

# Case One: Financing pressure index (FPI)

Quantify how the semantic content of a company’s news flow evolves in a way that can signal increasing likelihood of a financing announcement in the next 2 weeks.

This index becomes:

* a predictive feature for your financing-announcement model
* a managerial signal for early warning
* a research finding showing how communication patterns change



### Approach A: Embedding drift from the baseline 
**Steps:**

1. Step One: Cluster all news embeddings (KMeans, spectral clustering, HDBSCAN, etc.), before clustering the data, make use of dimensionality reduction techniques
    * t-SNE 
    * or UMAP
2. Step Two: Identify the baseline communication cluster
3. Step Three: Compute the centroid of this cluster
4. Step Four: For each biweekly aggregated embedding: compute the cosine distance to this baseline centroid 
5. Step Five: You just computed the Financing pressure index 

(Step Six: Scale this feature)

In [ ]:
def draw_umap(u, n_components=3, title='UMAP, n_components: 3'):
    random.seed(1)
    fig = plt.figure()
    if n_components == 1:
        ax = fig.add_subplot(111)
        ax.scatter(u[:,0], range(len(u)), c=data)
    if n_components == 2:
        ax = fig.add_subplot(111)
        ax.scatter(u[:,0], u[:,1], c=data)
    if n_components == 3:
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(u[:,0], u[:,1], u[:,2], s=100)
    plt.title(title, fontsize=18)

def dim_reduction(embeddings, dimensionality_red: str):
    
    if dimensionality_red == 'tsne':
        # -----------------------------------
        # Dimensionality reduction: t-SNE
        # -----------------------------------
        # INIT t-SNE
        random.seed(1)
        tsne = TSNE(
            n_components=3, 
            perplexity=5, 
            max_iter=250
        )

        # FIT & TRANSFORM
        reduced_embeddings = tsne.fit_transform(embeddings)

    else: 
        # -----------------------------------
        # Dimensionality reduction: UMAP
        # -----------------------------------
        # INIT UMAP
        random.seed(1)
        reducer = umap.UMAP(
            n_neighbors=2, # Local view --> default is 15, since data points is limited for most companies, use 2
            min_dist= 0.1, # Default 
            n_components=3, # Default
            metric='cosine'
        )

        # FIT & TRANSFORM
        reduced_embeddings = reducer.fit_transform(embeddings)
        #draw_umap(u=reduced_embeddings, n_components=3)

    return reduced_embeddings

def clustering(embeddings, periods, announcements):
    # INIT
    random.seed(1)
    dbscan = DBSCAN(eps=2, min_samples=2)
    labels = dbscan.fit_predict(embeddings)

    df = pd.DataFrame({
        'biweekly_period': [str(p) for p in periods],
        'label' : [lab for lab in labels], 
        'avg_finbert_array' : [comp for comp in dbscan.components_], 
        'next_announcement': [a for a in announcements]
    })

    return df

def compute_centroid(df):
    max_label = df.groupby(by='label') \
                  .count() \
                  .reset_index() \
                  .sort_values('avg_finbert_array', ascending=False) \
                  .reset_index() \
                  .head(1)['label'][0]
    max_label_embedding = mean_array(list(df[df['label'] == max_label]['avg_finbert_array']))

    return max_label, max_label_embedding

def compute_PFI(df, dimensionality_red: str): 
    # -----------------------------------
    # INIT  
    # -----------------------------------
    rows = df.select(["biweekly_period", "next_announcement", "avg_finbert_array"])
    rows = rows.collect()

    announcements = np.array([row['next_announcement'] for row in rows])
    periods = np.array([row["biweekly_period"] for row in rows])
    embeddings = np.array([row["avg_finbert_array"] for row in rows])

    # -----------------------------------
    # DIMENSIONALITY REDUCTION
    # -----------------------------------
    reduced_embeddings = dim_reduction(embeddings, dimensionality_red)

    # -----------------------------------
    # CLUSTERING 
    # -----------------------------------
    clustering_df = clustering(reduced_embeddings, periods, announcements)
    
    # -----------------------------------
    # CENTROID OF MAIN CLUSTER  
    # -----------------------------------
    centroid, embedding_centroid = compute_centroid(clustering_df)

    #print(f'Baseline cluster: cluster {centroid}, with embedding = {embedding_centroid}')

    # -----------------------------------
    # COSINE DISTANCE TO BASELINE CENTROID 
    # -----------------------------------
    clustering_df['FPI'] = clustering_df['avg_finbert_array'].apply(lambda x: cosine_distances([x], [embedding_centroid])[0][0])

    
    return clustering_df

def main(list_dfs): 

    dfs_with_PFI = []

    for df in list_dfs:
        print(df)

        try: 
            # COMPUTE FPI
            df_FPI = compute_PFI(df, dimensionality_red='umap')[['biweekly_period', 'FPI']]
            # JOIN TO SPARK DF
            df_FPI = spark.createDataFrame(df_FPI)
            joined_df = df.join(df_FPI, on="biweekly_period", how="inner")
        except:
            df = df.withColumn('FPI', F.lit(np.nan))
            joined_df = df
            print('Not Enough Articles Found')

        

        # APPEND 
        dfs_with_PFI.append(joined_df)

    return dfs_with_PFI


In [ ]:
if __name__ == "__main__":
    # Make sure detijd_final is a list (wrap it if it's a single DataFrame)
    if not isinstance(dfs3, list):
        dfs3 = [dfs3]
    
    dfs4 = main(dfs3)
    print(f"✅ Success! Processed {len(result)} DataFrames")
    dfs4[0].show()

DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]
DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]
DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, av

DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]
DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]
DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, av

... (10 outputs left out)

DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


DataFrame[biweekly_period: string, next_announcement: int, n_artices: bigint, avg_num_words_title: double, avg_num_words_content: double, avg_polarity_title: double, avg_polarity_content: double, avg_subjectivity_title: double, avg_subjectivity_content: double, avg_ollama: double, std_ollama: double, min_ollama: string, max_ollama: string, avg_finbert_array: array<double>, avg_finbert_vec: vector]


In [ ]:
'''reduced_embeddings = compute_PFI(dfs_company_websites[0], 'umap')
# Plot reduced embeddings
plt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], alpha=0.5)
plt.title("Embedding Space Visualization")
plt.show()'''

'reduced_embeddings = compute_PFI(dfs_company_websites[0], \'umap\')\n# Plot reduced embeddings\nplt.scatter(reduced_embeddings[:, 0], reduced_embeddings[:, 1], alpha=0.5)\nplt.title("Embedding Space Visualization")\nplt.show()'

We choose UMAP as the used technique since it is far more stable than t-SNE for our dataset

In [ ]:
'''embeddings = compute_PFI(dfs_company_websites[0], 'umap')

list_eps = [0.25, 0.4, 0.5, 0.75, 1, 1.5, 2, 2.5, 3, 4]

for eps in list_eps:
    dbscan = DBSCAN(eps=eps, min_samples=2)
    labels = dbscan.fit_predict(embeddings)
    
    try: 
        score = silhouette_score(embeddings, labels)
        print(f'the clustering with eps equal to {eps} has score: {score}')

    except Exception as e:
        print(e)'''

Number of labels is 1. Valid values are 2 to n_samples - 1 (inclusive)
the clustering with eps equal to 0.4 has score: -0.11835316568613052
the clustering with eps equal to 0.5 has score: -0.16199739277362823
the clustering with eps equal to 0.75 has score: 0.6199246048927307
the clustering with eps equal to 1 has score: 0.7902548909187317
the clustering with eps equal to 1.5 has score: 0.7902548909187317
the clustering with eps equal to 2 has score: 0.7902548909187317
the clustering with eps equal to 2.5 has score: 0.7902548909187317
the clustering with eps equal to 3 has score: 0.7902548909187317
the clustering with eps equal to 4 has score: 0.9064297676086426


after some iterative running of this code we have come to the conclusion that the ideal value for eps with the use of UMAP as the dimensionality reduction technique is: **EPS = 2**

In [ ]:
dfs_company_websites[0].show()

+---------------+-----------------+---------+-------------------+---------------------+-------------------+--------------------+----------------------+------------------------+----------+----------+----------+----------+--------------------+--------------------+--------------------+
|biweekly_period|next_announcement|n_artices|avg_num_words_title|avg_num_words_content| avg_polarity_title|avg_polarity_content|avg_subjectivity_title|avg_subjectivity_content|avg_ollama|std_ollama|min_ollama|max_ollama|   avg_finbert_array|     avg_finbert_vec|                 FPI|
+---------------+-----------------+---------+-------------------+---------------------+-------------------+--------------------+----------------------+------------------------+----------+----------+----------+----------+--------------------+--------------------+--------------------+
|            P33|                1|        1|                9.0|                188.0|                0.0| 0.08928571428571427|                   0

### Approach B: Embedding drift from financing events
**Steps:**

1. Step One: For all historical financing events, compute the average embedding of the 2 weeks before the financing. This creates the financing Signature Vector
2. For each NEW 2-week window, compute the cosine similarity to this signature. With new we mean that the computed distance is only to the average embedding pf past financing events.

In [ ]:
df = compute_PFI(dfs_company_websites[0], 'umap')[['biweekly_period', 'avg_finbert_array', 'next_announcement']]

25/12/10 20:15:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
{"ts": "2025-12-10 20:15:57.746", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `avg_finbert_array` cannot be resolved. Did you mean one of the following? [`content_lemma`, `period_start`, `content`, `ticker`, `title_lemma`]. SQLSTATE: 42703", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o19618.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `avg_finbert_array` cannot be resolved. Did you

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `avg_finbert_array` cannot be resolved. Did you mean one of the following? [`content_lemma`, `period_start`, `content`, `ticker`, `title_lemma`]. SQLSTATE: 42703;
'Project [biweekly_period#171628, next_announcement#171668, 'avg_finbert_array]
+- Join RightOuter, ((date#171677 >= period_start#171629) AND (date#171677 <= period_end#171630))
   :- Project [content#171707, date#171677, title#171709, url#171673, ticker#171675, content_lemma#171711, title_lemma#171713, title_tokenized#171714, content_tokenized#171715, num_words_title#171716, size(content_tokenized#171715, false) AS num_words_content#171717]
   :  +- Project [content#171707, date#171677, title#171709, url#171673, ticker#171675, content_lemma#171711, title_lemma#171713, title_tokenized#171714, content_tokenized#171715, size(title_tokenized#171714, false) AS num_words_title#171716]
   :     +- Project [content#171707, date

In [ ]:
df['period'] = df['biweekly_period'].apply(lambda x: int(x[1:3]) if len(x) == 3 else int(x[1]))

df = df.sort_values(by='period')
df.head()

,biweekly_period,label,avg_finbert_array,next_announcement,cosine_distance,period
14,P0,4,"[6.6243615, -3.559074, -0.4824599]",1,0.673934,0
21,P1,2,"[1.3701513, 11.001975, -2.3984566]",1,1.088220,1
3,P2,1,"[7.7649975, 0.8084698, 14.453873]",1,0.000117,2
11,P3,3,"[-4.3497705, 5.3288093, -1.1440201]",1,1.397800,3
18,P4,0,"[-2.3181648, 5.3253045, 6.53473]",0,0.431503,4
22,P5,8,"[5.9859643, -5.8150935, -0.63323766]",0,0.769641,5
33,P6,4,"[6.518327, -3.4433758, -0.36827263]",1,0.659297,6
6,P7,3,"[-4.3913474, 5.287279, -1.1854969]",1,1.406005,7
8,P8,4,"[6.708971, -3.6436462, -0.56846714]",1,0.684440,8
34,P9,1,"[7.023502, 0.7436273, 14.517363]",0,0.000382,9


In [ ]:
def add_last_announcement_embedding(df):
    last_embedding = None
    emb_last_announcement = []

    for _, row in df.iterrows():
        emb_last_announcement.append(last_embedding)

        # Update last embedding if current row is announcement
        if row['next_announcement'] == 1:
            last_embedding = row['avg_finbert_array']

    df['emb_last_announcement'] = emb_last_announcement
    return df

df = add_last_announcement_embedding(df)
df.head()

,biweekly_period,label,avg_finbert_array,next_announcement,cosine_distance,period,emb_last_announcement
14,P0,4,"[6.6243615, -3.559074, -0.4824599]",1,0.673934,0,None
21,P1,2,"[1.3701513, 11.001975, -2.3984566]",1,1.088220,1,"[6.6243615, -3.559074, -0.4824599]"
3,P2,1,"[7.7649975, 0.8084698, 14.453873]",1,0.000117,2,"[1.3701513, 11.001975, -2.3984566]"
11,P3,3,"[-4.3497705, 5.3288093, -1.1440201]",1,1.397800,3,"[7.7649975, 0.8084698, 14.453873]"
18,P4,0,"[-2.3181648, 5.3253045, 6.53473]",0,0.431503,4,"[-4.3497705, 5.3288093, -1.1440201]"


In [ ]:
def compute_financing_sim(row):
    if row['emb_last_announcement'] is None:
        return np.nan  # or another placeholder
    return cosine_similarity(
        np.array(row['avg_finbert_array']).reshape(1, -1),
        np.array(row['emb_last_announcement']).reshape(1, -1)
    )[0][0]

df['Financing_sim'] = df.apply(compute_financing_sim, axis=1)
df.head(10)

,biweekly_period,label,avg_finbert_array,next_announcement,cosine_distance,period,emb_last_announcement,Financing_sim
14,P0,4,"[6.6243615, -3.559074, -0.4824599]",1,0.673934,0,None,NaN
21,P1,2,"[1.3701513, 11.001975, -2.3984566]",1,1.088220,1,"[6.6243615, -3.559074, -0.4824599]",-0.338375
3,P2,1,"[7.7649975, 0.8084698, 14.453873]",1,0.000117,2,"[1.3701513, 11.001975, -2.3984566]",-0.081210
11,P3,3,"[-4.3497705, 5.3288093, -1.1440201]",1,1.397800,3,"[7.7649975, 0.8084698, 14.453873]",-0.401592
18,P4,0,"[-2.3181648, 5.3253045, 6.53473]",0,0.431503,4,"[-4.3497705, 5.3288093, -1.1440201]",0.508246
22,P5,8,"[5.9859643, -5.8150935, -0.63323766]",0,0.769641,5,"[-4.3497705, 5.3288093, -1.1440201]",-0.964680
33,P6,4,"[6.518327, -3.4433758, -0.36827263]",1,0.659297,6,"[-4.3497705, 5.3288093, -1.1440201]",-0.899183
6,P7,3,"[-4.3913474, 5.287279, -1.1854969]",1,1.406005,7,"[6.518327, -3.4433758, -0.36827263]",-0.901196
8,P8,4,"[6.708971, -3.6436462, -0.56846714]",1,0.684440,8,"[-4.3913474, 5.287279, -1.1854969]",-0.899941
34,P9,1,"[7.023502, 0.7436273, 14.517363]",0,0.000382,9,"[6.708971, -3.6436462, -0.56846714]",0.292554


# To Do
* avg, min, max, std of ollama sentiment **OK**
* maybe check other models (ollama, or others)
* TextBlob **OK, except for word2vec**
* other preprocessing steps **OK**
    - remove urls, numbers (anything stock price related, financial results, etc.), stock tickers,
    - stemming/lemmatization 
        - create a column in the dataframe with lemmatized content. (only for TextBlob)
    - removing special characters
* UDF to get an average embedding (avg of multiple arrays) **OK**
* think about other use case for the embeddings
    - Financing pressure index
    - Semantic change detection
    - Embedding-Based “Narrative Similarity” to Past Financing Events
    - (Clustering news in themes)
* topic modeling and summerization

* Creating a basetable (don't forget the ticker and period for later merging) **DOING**
* Merge with augmented data, whenever dependent variable was added. in stead of biweekly_data.parquet

* Small modeling case **COPY PASTE (& ADD XGBOOST)**
